# TF/matplotlib Kaggle Verification (Diagnostic Only) — revision 2

> **Revision 2 (2026-09-25).** Revision 1 ran directly in Kaggle's system Python
> (3.12.13) and hit two environment artifacts — a stale `sys.modules` cache reporting
> matplotlib 3.10.0 despite pip installing 3.9.0, and a numpy ABI break from mixing the
> governed `numpy==1.26.4` pin onto Kaggle's numpy-2.x-linked preinstalled packages — plus
> one genuine, expected, project-level finding: `tests/test_release_hashes.py` fails to
> even COLLECT on Kaggle, by design, until **W-6 step 8** (the in-Kaggle write-durability
> measurement, already tracked as owed in `foundation/code-generation/code-summary.md`)
> characterises the platform. Revision 2 fixes the first two (isolated venv at the
> governed Python 3.11, subprocess-based version checks, mirroring the pattern already
> used by `kaggle_iri2016_verification.ipynb`) and reports the third distinctly rather
> than papering over it.

**Purpose.** Close the TE §8.1 "both-platform" (Kaggle AND local) condition on the
`tensorflow==2.21.0` (D-36) and `matplotlib==3.9.0` (Recommendation 38) pins in
`requirements.txt`. Item 2 of `build-and-test`'s CONDITIONAL PASS closed the LOCAL half
of this check on 2026-09-25; this notebook is the Kaggle half, per TC-03g's mandate.

**Scope, deliberately narrow.** Installs the exact governed pin surface into an isolated
venv at the governed Python version, verifies `tensorflow`/`matplotlib` import at the
exact pinned versions (via subprocess, not in-kernel import), and runs the §18.3
critical test set — selection (b), D-69. Does **NOT** run either walking-skeleton
fixture (item 3, still open). Does **NOT** touch GNSS/VTEC target data, fit any model,
or open the locked December test set. Diagnostic only.

**How to use.** Import into a new Kaggle Notebook, CPU-only accelerator, internet **on**.
Run all cells top to bottom. Download
`/kaggle/working/tf_matplotlib_verification_bundle.zip` from the output pane when done.

In [ ]:
import datetime as dt
import json
import platform
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

REPO_URL = "https://github.com/Kimrza/Thesis_toshkari.git"
PINNED_COMMIT = "7357f3504466dd883249eefd9a7064267997e7a3"

REPO_DIR = Path("/kaggle/working/Thesis_toshkari")
VENV_DIR = Path("/kaggle/working/tf_matplotlib_venv")
BUNDLE_DIR = Path("/kaggle/working/tf_matplotlib_verification_bundle")
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

report = {
    "notebook": "kaggle_tf_matplotlib_pin_verification.ipynb",
    "notebook_revision": 3,
    "purpose": "TE \u00a78.1 both-platform check for tensorflow==2.21.0 (D-36) and "
               "matplotlib==3.9.0 (Rec 38); build-and-test item 2, Kaggle half",
    "pinned_commit_requested": PINNED_COMMIT,
    "generated_at_utc": None,
}

def run(cmd, timeout=1800, **kw):
    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, **kw)
    return {
        "cmd": cmd if isinstance(cmd, str) else " ".join(cmd),
        "returncode": proc.returncode,
        "stdout_tail": proc.stdout[-4000:],
        "stderr_tail": proc.stderr[-4000:],
    }

def write_bundle():
    report["generated_at_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
    (BUNDLE_DIR / "report.json").write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
    zip_path = Path("/kaggle/working/tf_matplotlib_verification_bundle.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in BUNDLE_DIR.rglob("*"):
            if p.is_file():
                zf.write(p, p.relative_to(BUNDLE_DIR.parent))
    return zip_path

def save_and_stop(reason):
    report["ok"] = False
    report["stop_reason"] = reason
    zip_path = write_bundle()
    print("STOPPED:", reason)
    print("Bundle written to:", zip_path)
    raise SystemExit(1)


## Step 1 — Inspect Kaggle's actual Python/platform

Kaggle's system Python is very likely NOT the governed 3.11.16 (TS-01/TC-03d) — this
cell records what it actually is. Step 2 creates an isolated venv at 3.11 regardless,
so nothing below depends on the kernel's own Python matching.

In [ ]:
runtime = {
    "python_version": sys.version,
    "python_version_info": list(sys.version_info),
    "python_executable": sys.executable,
    "platform_platform": platform.platform(),
    "platform_machine": platform.machine(),
    "platform_system": platform.system(),
    "in_kaggle": Path("/kaggle").is_dir(),
}
report["runtime"] = runtime
print(json.dumps(runtime, indent=2))


## Step 2 — Create an isolated venv at the governed Python 3.11

Mirrors `kaggle_iri2016_verification.ipynb`'s rung strategy exactly (virtualenv against
an existing python3.11, else a uv-managed standalone CPython 3.11, else apt-get as a
last resort) — never installs the governed pins into Kaggle's own system Python, which
is what caused revision 1's numpy ABI break against Kaggle's pre-installed, numpy-2.x-
linked package set.

In [ ]:
venv_python_source = None
if sys.version_info[:2] == (3, 11):
    venv_python_source = sys.executable
    strategy = "kernel_python_is_3.11_isolate_via_venv"
else:
    which = shutil.which("python3.11")
    if which:
        venv_python_source = which
        strategy = "found_existing_python3.11_isolate_via_venv"
    else:
        strategy = "no_system_python3.11_use_uv_managed_3.11"

report["environment_strategy"] = {
    "kernel_python_minor": list(sys.version_info[:2]),
    "strategy": strategy,
    "venv_python_source": venv_python_source,
}
print(json.dumps(report["environment_strategy"], indent=2, default=str))
write_bundle()


In [ ]:
env_logs = report.setdefault("installation", {}).setdefault("isolated_env_creation", [])
venv_python = str(VENV_DIR / "bin" / "python")

def env_ready():
    return Path(venv_python).is_file() and run([venv_python, "-m", "pip", "--version"])["returncode"] == 0

mechanism = None

def fresh():
    if VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)

def attempt(label, cmd, **kw):
    env_logs.append({"attempt": label, **run(cmd, **kw)})
    write_bundle()

# Rung 1: virtualenv against the image's own python3.11 (only if one exists).
if venv_python_source is not None:
    fresh()
    attempt("rung1: pip install virtualenv into kernel python",
            [sys.executable, "-m", "pip", "install", "--quiet", "--no-input", "virtualenv"], timeout=300)
    attempt("rung1: virtualenv -p python3.11",
            [sys.executable, "-m", "virtualenv", "-p", venv_python_source, str(VENV_DIR)], timeout=300)
    if env_ready():
        mechanism = "virtualenv against the image python3.11"

# Rung 2: a uv-managed CPython 3.11 (complete standalone interpreter, immutable release).
if mechanism is None:
    fresh()
    attempt("rung2: pip install uv into kernel python",
            [sys.executable, "-m", "pip", "install", "--quiet", "--no-input", "uv==0.12.17"], timeout=300)
    uv_env = {"UV_PYTHON_INSTALL_DIR": "/kaggle/working/uv_python", "UV_CACHE_DIR": "/kaggle/working/uv_cache"}
    attempt("rung2: uv python install cpython-3.11.16",
            [sys.executable, "-m", "uv", "python", "install", "cpython-3.11.16"], timeout=600, env=uv_env)
    attempt("rung2: uv venv --seed --python 3.11.16",
            [sys.executable, "-m", "uv", "venv", "--seed", "--python", "cpython-3.11.16", str(VENV_DIR)], timeout=300, env=uv_env)
    if env_ready():
        mechanism = "uv 0.12.17 managed cpython-3.11.16 (python-build-standalone release) + uv venv --seed"

# Rung 3 (last resort): Debian's python3.11-venv, non-interactive, bounded.
if mechanism is None and venv_python_source is not None:
    fresh()
    attempt("rung3: apt-get install python3.11-venv (noninteractive)",
            ["bash", "-lc", "export DEBIAN_FRONTEND=noninteractive; apt-get update -qq && apt-get install -y -qq --no-install-recommends python3.11-venv"],
            timeout=240)
    attempt("rung3: python3.11 -m venv", [venv_python_source, "-m", "venv", str(VENV_DIR)], timeout=300)
    if env_ready():
        mechanism = "stdlib venv after apt-get python3.11-venv"

if mechanism is None:
    summary = chr(10).join(
        f"  - {e['attempt']}: exit={e['returncode']} stderr_tail={(e.get('stderr_tail') or e.get('stdout_tail') or '')[-400:].strip()!r}"
        for e in env_logs
    )
    save_and_stop("no isolated Python 3.11 environment could be created: every rung failed. Per-attempt summary:" + chr(10) + summary)

venv_info = run([venv_python, "-c", "import sys, platform; print(sys.version); print(platform.platform())"])
report["environment_strategy"]["venv_python_version"] = venv_info["stdout_tail"].strip()
report["environment_strategy"]["isolated_env_mechanism"] = mechanism
write_bundle()
print(json.dumps(report["environment_strategy"], indent=2, default=str))


## Step 3 — Clone the repository at the pinned commit

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone = run(["git", "clone", "--no-tags", REPO_URL, str(REPO_DIR)])
report["clone"] = clone
if clone["returncode"] != 0:
    save_and_stop(f"git clone failed (exit {clone['returncode']}): {clone['stderr_tail']}")

checkout = run(["git", "-C", str(REPO_DIR), "checkout", PINNED_COMMIT])
report["checkout"] = checkout
if checkout["returncode"] != 0:
    save_and_stop(f"git checkout of pinned commit failed: {checkout['stderr_tail']}")

actual_head = run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"])["stdout_tail"].strip()
report["actual_head"] = actual_head
assert actual_head == PINNED_COMMIT, f"checked out {actual_head}, expected {PINNED_COMMIT}"
print("Checked out:", actual_head)


## Step 4 — Install the exact governed pin surface INTO THE ISOLATED VENV

`requirements.txt` — the single governed pin surface (TE §13.1) — installed into the
clean venv created in Step 2, never into Kaggle's system Python.

In [ ]:
install_reqs = run(
    [venv_python, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")],
    timeout=1800,
)
report["install_requirements"] = install_reqs
if install_reqs["returncode"] != 0:
    save_and_stop(f"pip install -r requirements.txt failed in the isolated venv: {install_reqs['stderr_tail']}")

freeze = run([venv_python, "-m", "pip", "freeze"])
report["pip_freeze"] = freeze["stdout_tail"]
print("Install complete in isolated venv.")


## Step 5 — Verify tensorflow and matplotlib import at the exact pinned versions

Run as a SEPARATE subprocess (not an in-kernel `import`) so a stale `sys.modules`
entry from Kaggle's own kernel startup (revision 1's matplotlib false-negative) cannot
mask the real, freshly-installed version.

In [ ]:
PIN_CHECK_SCRIPT = r'''
import json
result = {}
try:
    import tensorflow as tf
    result["tensorflow"] = {"imported": True, "version": tf.__version__}
    assert tf.__version__ == "2.21.0", f"expected 2.21.0, got {tf.__version__}"
    from tensorflow import keras
    m = keras.Sequential([keras.layers.LSTM(4, input_shape=(3, 2)), keras.layers.Dense(1)])
    m.compile(optimizer="adam", loss="mse")
    result["tensorflow"]["keras_lstm_smoke"] = "ok"
except Exception as exc:
    result["tensorflow"] = {"imported": False, "error": str(exc)}

try:
    import matplotlib
    result["matplotlib"] = {"imported": True, "version": matplotlib.__version__}
    assert matplotlib.__version__ == "3.9.0", f"expected 3.9.0, got {matplotlib.__version__}"
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots()
    ax.plot([0, 1], [0, 1])
    plt.close(fig)
    result["matplotlib"]["figure_smoke"] = "ok"
except Exception as exc:
    result["matplotlib"] = {"imported": False, "error": str(exc)}

print(json.dumps(result))
'''

pin_check_path = BUNDLE_DIR / "pin_check_inner.py"
pin_check_path.write_text(PIN_CHECK_SCRIPT, encoding="utf-8")

# Kaggle's kernel sets MPLBACKEND to its own Jupyter-only inline backend
# ("module://matplotlib_inline.backend_inline"), which does not exist in our clean
# venv and would otherwise be inherited by this subprocess and break both imports
# (revision 1/2 finding, 2026-09-25). Override it explicitly rather than inheriting.
pin_check_env = dict(**__import__("os").environ)
pin_check_env.pop("MPLBACKEND", None)
pin_check_env["MPLBACKEND"] = "Agg"
pin_check_run = run([venv_python, str(pin_check_path)], timeout=300, env=pin_check_env)
report["pin_check_process"] = pin_check_run
try:
    pin_check = json.loads(pin_check_run["stdout_tail"].strip().splitlines()[-1])
except Exception as exc:
    save_and_stop(f"pin-check subprocess did not print parseable JSON (exit {pin_check_run['returncode']}): {exc}; stderr: {pin_check_run['stderr_tail']}")

report["pin_check"] = pin_check
print(json.dumps(pin_check, indent=2))
write_bundle()


## Step 6 — Run the §18.3 critical test set (selection (b), D-69)

Run inside the isolated venv, `PYTHONHASHSEED=0`. **Expected finding, not a notebook
bug:** `test_release_hashes.py` will fail to COLLECT with `LockedTestError` naming
"W-6 step 8" — Kaggle's write-durability semantics are uncharacterised, and the
locked-test guard fails closed by design (`src/data/locked_test.py`). This cell runs
the other nine modules separately so a real result for them isn't lost to that one
module's collection error, and records the tenth's refusal distinctly.

In [ ]:
import os

env = dict(**os.environ)
env["PYTHONHASHSEED"] = "0"

crit_modules_no_release_hashes = [
    "tests/test_prepared_target_schema.py",
    "tests/test_feature_availability.py",
    "tests/test_iri_denial.py",
    "tests/test_split_embargo.py",
    "tests/test_train_only_transforms.py",
    "tests/test_common_masks.py",
    "tests/test_checkpoint_restore.py",
    "tests/test_bootstrap.py",
    "tests/test_locked_test_guard.py",
]
junit_path = BUNDLE_DIR / "crit_kaggle_nine_of_ten.xml"

crit_run = subprocess.run(
    [venv_python, "-m", "pytest", *crit_modules_no_release_hashes, f"--junitxml={junit_path}", "-q"],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
    timeout=1800,
    env=env,
)
report["critical_set_run_nine_of_ten"] = {
    "returncode": crit_run.returncode,
    "stdout_tail": crit_run.stdout[-6000:],
    "stderr_tail": crit_run.stderr[-3000:],
}
print(crit_run.stdout[-3000:])
print("Nine-of-ten exit code:", crit_run.returncode)

# The tenth module, run alone, so its collection error is captured cleanly and
# distinctly rather than aborting the other nine.
release_hashes_junit = BUNDLE_DIR / "test_release_hashes_kaggle.xml"
release_hashes_run = subprocess.run(
    [venv_python, "-m", "pytest", "tests/test_release_hashes.py", f"--junitxml={release_hashes_junit}", "-q"],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
    timeout=300,
    env=env,
)
report["test_release_hashes_run"] = {
    "returncode": release_hashes_run.returncode,
    "stdout_tail": release_hashes_run.stdout[-3000:],
    "stderr_tail": release_hashes_run.stderr[-1000:],
    "expected_blocker": "LockedTestError naming W-6 step 8 is the CORRECT, EXPECTED result "
                         "until the in-Kaggle durability measurement lands; any other error "
                         "is a real finding.",
}
print(release_hashes_run.stdout[-1500:])
write_bundle()


## Step 7 — Finalize the diagnostic bundle

Zipped to `/kaggle/working/tf_matplotlib_verification_bundle.zip` — download that
file from the Kaggle output pane and bring it back.

In [ ]:
import re

def parse_counts(path):
    if not path.exists():
        return None
    text = path.read_text(encoding="utf-8")
    m = re.search(r'tests="(\d+)" errors="(\d+)" failures="(\d+)" skipped="(\d+)"', text)
    if not m:
        return None
    return {"tests": int(m.group(1)), "errors": int(m.group(2)), "failures": int(m.group(3)), "skipped": int(m.group(4))}

report["critical_set_counts_nine_of_ten"] = parse_counts(junit_path)
report["test_release_hashes_counts"] = parse_counts(release_hashes_junit)

nine_ok = report.get("critical_set_run_nine_of_ten", {}).get("returncode") == 0
pins_ok = pin_check.get("tensorflow", {}).get("imported") and pin_check.get("matplotlib", {}).get("imported")
release_hashes_blocked_as_expected = "W-6 step 8" in (
    report.get("test_release_hashes_run", {}).get("stdout_tail", "")
)

report["ok"] = nine_ok and pins_ok
report["release_hashes_blocked_as_expected"] = release_hashes_blocked_as_expected
report["summary"] = {
    "pins_verified": pins_ok,
    "nine_of_ten_critical_modules_green": nine_ok,
    "test_release_hashes_status": "blocked_pending_W6_step8" if release_hashes_blocked_as_expected else "UNEXPECTED_RESULT_INVESTIGATE",
}

zip_path = write_bundle()
print("DONE. Download this file from the Kaggle output pane:", zip_path)
print(json.dumps(report["summary"], indent=2))
